# aerosearch — GPU batch indexer (Kaggle T4)

Runs the heavy understanding (Gemma 3 4B + Whisper large-v3 + e5) on a Kaggle GPU
against **MinIO + Postgres running on your Mac**, reachable over a tunnel.

It reads rows your bot inserted with a NULL embedding, understands + embeds them,
and writes the vectors back — so `/search` in your local bot works right after.

## Before running
1. **Enable GPU**: Notebook → Settings → Accelerator → *GPU T4 x2* (any T4 is fine).
2. **Expose your Mac's services** (Tailscale or `ngrok tcp 5432` / `ngrok tcp 9000`).
3. **Upload the repo as a Kaggle Dataset** (folder containing `pyproject.toml` +
   `aerosearch/`) and attach it, OR pip-install from your git remote.
4. **Add your HF token** as a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets) —
   Gemma is gated.
5. Fill in the tunnel endpoints + credentials in the **Config** cell below.

In [ ]:
# --- Install aerosearch + GPU deps ---
# Option A: from an attached dataset (folder with pyproject.toml + aerosearch/)
%pip install -q "/kaggle/input/aerosearch-src[gpu]"
# Option B: from a git remote instead (comment A, uncomment B)
# %pip install -q "git+https://github.com/<you>/aerosearch@main#egg=aerosearch[gpu]"
print('installed')

In [ ]:
# --- Config: point at the tunnel to your Mac ---
import os

# Postgres over the tunnel (Tailscale IP, or ngrok tcp host:port)
PG_HOST = "100.x.y.z"      # e.g. Tailscale IP, or 0.tcp.ngrok.io
PG_PORT = 5432             # ngrok gives a random high port — use it here
PG_USER = "postgres"
PG_PASSWORD = "postgres"
PG_DB = "aerosearch"
DSN = f"postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"

# MinIO over the tunnel
MINIO_HOST = "100.x.y.z"   # same Tailscale IP, or ngrok host
MINIO_PORT = 9000
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
MINIO_BUCKET = "media"

# Models (GPU): the 4-bit Gemma 3 4B captioner + strong multilingual Whisper.
os.environ["AEROSEARCH_CAPTION_MODEL"] = "google/gemma-3-4b-it"
os.environ["AEROSEARCH_CAPTION_4BIT"] = "1"
os.environ["AEROSEARCH_WHISPER_MODEL"] = "large-v3"
os.environ["AEROSEARCH_EMBED_MODEL"] = "intfloat/multilingual-e5-large"

# HF token (gated Gemma) from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Set HF_TOKEN via Add-ons -> Secrets:", e)

print("DSN:", DSN.replace(PG_PASSWORD, "***"))

In [ ]:
# --- Quick connectivity check before loading big models ---
import socket
for host, port, name in [(PG_HOST, PG_PORT, "postgres"), (MINIO_HOST, MINIO_PORT, "minio")]:
    s = socket.socket(); s.settimeout(5)
    try:
        s.connect((host, port)); print(f"OK  {name} reachable at {host}:{port}")
    except Exception as e:
        print(f"XX  {name} NOT reachable at {host}:{port} -> {e}")
    finally:
        s.close()

In [ ]:
# --- Build the batch indexer with real GPU backends ---
import torch
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

from aerosearch.config import load_config
from aerosearch.factory import build_captioner, build_embedder, build_transcriber
from aerosearch.store import get_pgvector_store
from aerosearch.store.minio_source import MinioDownloader
from aerosearch.batch import BatchIndexer

cfg = load_config()
store = get_pgvector_store(DSN, table="media", dim=cfg.model.embed_dim)
downloader = MinioDownloader(
    endpoint=f"{MINIO_HOST}:{MINIO_PORT}",
    access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY,
    bucket=MINIO_BUCKET, secure=False,
)
batch = BatchIndexer(
    cfg,
    captioner=build_captioner(cfg, "real"),
    transcriber=build_transcriber(cfg, "real"),
    embedder=build_embedder(cfg, "real"),
    store=store,
    downloader=downloader,
)
print("ready")

In [ ]:
# --- Run the batch: understand + embed pending rows, write vectors back ---
import asyncio, logging
logging.basicConfig(level=logging.INFO)

async def run():
    n = await batch.run(limit=200)   # process up to 200 pending rows
    await store.close()
    return n

indexed = await run() if asyncio.get_event_loop().is_running() else asyncio.run(run())
print(f"indexed {indexed} row(s)")

In [ ]:
# --- Sanity search over what was just indexed (uses your remote Postgres) ---
from aerosearch.pipeline import search as run_search

store2 = get_pgvector_store(DSN, table="media", dim=cfg.model.embed_dim)
embedder = build_embedder(cfg, "real")

async def do_search(q):
    hits = await run_search(q, embedder, store2, k=5)
    await store2.close()
    for i, h in enumerate(hits, 1):
        print(f"{i}. {h.score:.3f}  {h.media_type}  {h.source}  | {h.document[:80]!r}")

await do_search("речь политика")